In [1]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, DataCollatorForLanguageModeling ,Trainer,TrainingArguments
from datasets import load_dataset
from pprint import pprint

2025-10-26 23:28:00.781393: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-26 23:28:00.828390: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-26 23:28:02.035031: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Causal language modeling 

Causal language modeling predicts the next token in a sequence of tokens, and the model can only attend to tokens on the left. This means the model cannot see future tokens (this is what the mask do in the decoder multi-head-attention)

In [2]:
#Load the dataset and split it 
eli5_ds = load_dataset("dany0407/eli5_category", split="train[:5000]").train_test_split(test_size=0.2)

#Initialize the tokenizer
tokenizer_model = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)

In [3]:
#Let's flatten the dataset to better have access to the imbricated columns
eli5_ds = eli5_ds.flatten()
eli5_ds.column_names
eli5_ds['train'][0]

{'q_id': '774lu6',
 'title': 'Why is titanium flammable?',
 'selftext': 'Specifically titanium shavings',
 'category': 'Chemistry',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['doj4mp3', 'doj25au', 'doj5t6v'],
 'answers.text': ["This is not really something special about titanium. Many materials have pretty much the same thing going on. Metals like titanium can oxidize. When that happens to iron we call it rust. With most metal objects they automatically form a thin layer of oxidized material on the surface. If you scratch that layer you expose the unoxidized metal which on contact with oxygen oxidizes. The problem comes when the protective layer can not form quickly enough. If you create lots of metal shavings you end up with a really big amount of surface area per mass of metal and all that surface area is exposed to oxygen in the air. A single spark under this circumstances can set it all aflame and lead to a small (or not so small) explosion. This can also happen with mate

In [4]:
def preprocess(ds):
    return tokenizer([" ".join(x) for x in ds['answers.text']]) #We split every character because we use a letter-level-tokenizer

#Let's apply this function to our dataset
tokenized_eli5_ds = eli5_ds.map(
    function= preprocess,
    batched=True,
    num_proc=4, 
    remove_columns= eli5_ds['train'].column_names
)

liste =  tokenized_eli5_ds["train"]
print(liste['input_ids'])
print(len(liste['input_ids']))


    
# concatenated_examples = {k: sum(tokenized_eli5_ds["train"][0][k], []) for k in tokenized_eli5_ds["train"][0].keys()}

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1374 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1385 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (3145 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1501 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1058 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2071 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2896 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1801 > 1024). Running this sequence through the model will result in indexing errors


Column([[1212, 318, 407, 1107, 1223, 2041, 546, 41284, 13, 4650, 5696, 423, 2495, 881, 262, 976, 1517, 1016, 319, 13, 3395, 874, 588, 41284, 460, 18762, 1096, 13, 1649, 326, 4325, 284, 6953, 356, 869, 340, 17000, 13, 2080, 749, 6147, 5563, 484, 6338, 1296, 257, 7888, 7679, 286, 18762, 1143, 2587, 319, 262, 4417, 13, 1002, 345, 12692, 326, 7679, 345, 15651, 262, 555, 1140, 312, 1143, 6147, 543, 319, 2800, 351, 11863, 18762, 4340, 13, 383, 1917, 2058, 618, 262, 14153, 7679, 460, 407, 1296, 2952, 1576, 13, 1002, 345, 2251, 6041, 286, 6147, 427, 615, 654, 345, 886, 510, 351, 257, 1107, 1263, 2033, 286, 4417, 1989, 583, 2347, 286, 6147, 290, 477, 326, 4417, 1989, 318, 7362, 284, 11863, 287, 262, 1633, 13, 317, 2060, 9009, 739, 428, 5917, 460, 900, 340, 477, 257, 49621, 290, 1085, 284, 257, 1402, 357, 273, 407, 523, 1402, 8, 11278, 13, 770, 460, 635, 1645, 351, 5696, 326, 3588, 470, 6147, 475, 389, 379, 1551, 6454, 781, 6475, 540, 13, 5326, 8977, 460, 22818, 1165, 13, 317, 5863, 1672, 286, 3

We have something like:  
- **dataset = Dataset ( 'input_ids' : List [ List[int] ] , 'attention_mask' : List [ List[int] ] )**

**Example:**  

{  

  'input_ids': [  

    [101, 2023, 2003, 1037, 2742, 102, 0, 0, 0],  

    [101, 2054, 2003, 2023, 102, 0, 0, 0, 0],  

    [101, 1045, 2293, 2070, 2742, 102, 0, 0, 0]  

  ],  

  'attention_mask': [  

    [1, 1, 1, 1, 1, 1, 0, 0, 0],  

    [1, 1, 1, 1, 1, 1, 0, 0, 0],  

    [1, 1, 1, 1, 1, 1, 0, 0, 0]  
    
  ]  
}

In [5]:
block_size = 128

def group_texts(examples):
    """ 
    examples : Dataset
    """
    concat_examples= {k : sum(examples[k],[]) for k in examples.keys()}
    total_length = len(concat_examples["input_ids"])
    if total_length >= block_size:
        total_length = (total_length//block_size) * block_size
    #Split by chunks of block_size
    result = {k : [t[i : i + block_size] for i in range(0,total_length,block_size)] for k,t in concat_examples.items()}
    labels = result["input_ids"].copy()
    return result
    

In [6]:
#Let's apply this for ou eli5_ds

final_ds= tokenized_eli5_ds.map(group_texts,batched=True , num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
#This model does not have a pad token , so for this we give it the eos token 
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer,mlm=False)


model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args = training_args,
    data_collator=data_collator,
    train_dataset= final_ds["train"],
    eval_dataset= final_ds["test"],
    tokenizer= tokenizer
)

trainer.train()


/tmp/ipykernel_28248/3954705149.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 